[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dandi/example-notebooks/blob/master/000041/BuzsakiLab/watson16/watson16_alignment.ipynb)

## Installing requirements

The cell below installs every Python package needed to run this notebook, at fully pinned versions, using [`uv`](https://github.com/astral-sh/uv) for fast resolution. In Colab the cell is collapsed by default — click the ▶ button to run it.

In [ ]:
#@title Installing requirements (click ▶ to run) { display-mode: "form" }
# Colab provides Python 3.13. We install with `uv --system` because Colab's
# kernel runs outside a virtualenv. All versions (direct + transitive) are
# pinned below so the notebook is reproducible regardless of resolver drift.
!pip install -q uv
!uv pip install --system \
    "acres==0.5.0" \
    "aiohappyeyeballs==2.7.1" \
    "aiohttp==3.14.3" \
    "aiosignal==1.4.0" \
    "annotated-types==0.8.0" \
    "arrow==1.4.0" \
    "attrs==26.1.0" \
    "bids-validator-deno==3.0.2" \
    "bidsschematools==1.2.7" \
    "blessed==1.50.0" \
    "certifi==2026.7.22" \
    "cffi==2.1.1" \
    "charset-normalizer==3.4.9" \
    "ci-info==0.4.0" \
    "click==8.5.0" \
    "click-didyoumean==0.3.1" \
    "cloudpickle==3.1.2" \
    "contourpy==1.3.3" \
    "cryptography==50.0.1" \
    "cycler==0.12.1" \
    "dandi==0.81.0" \
    "dandischema==0.14.0" \
    "deno==2.9.8" \
    "dnspython==2.9.1" \
    "donfig==0.8.1.post1" \
    "email-validator==2.3.0" \
    "etelemetry==0.3.1" \
    "fasteners==0.20" \
    "fonttools==4.64.0" \
    "fqdn==1.5.1" \
    "frozenlist==1.8.0" \
    "fscacher==0.5.0" \
    "fsspec==2025.12.0" \
    "google-crc32c==1.8.0" \
    "h5py==3.16.0" \
    "hdmf==6.2.0" \
    "humanize==4.16.0" \
    "idna==3.19" \
    "interleave==0.3.0" \
    "isodate==0.7.2" \
    "isoduration==20.11.0" \
    "jaraco-classes==3.4.0" \
    "jaraco-context==6.1.2" \
    "jaraco-functools==4.6.0" \
    "jeepney==0.9.0" \
    "jinxed==2.1.0" \
    "joblib==1.6.0" \
    "jsonpointer==3.1.1" \
    "jsonschema==4.26.0" \
    "jsonschema-specifications==2025.9.1" \
    "keyring==25.7.0" \
    "keyrings-alt==5.0.2" \
    "kiwisolver==1.5.1" \
    "matplotlib==3.10.0" \
    "ml-dtypes==0.6.0" \
    "more-itertools==10.8.0" \
    "multidict==6.7.1" \
    "natsort==8.4.0" \
    "numcodecs==0.17.1" \
    "numpy==2.1.3" \
    "nwbinspector==0.7.2" \
    "packaging==26.3" \
    "pandas==2.2.3" \
    "pillow==11.3.0" \
    "platformdirs==4.11.7" \
    "propcache==0.5.2" \
    "pycparser==3.0" \
    "pycryptodomex==3.23.0" \
    "pydantic==2.13.5" \
    "pydantic-core==2.46.5" \
    "pydantic-settings==2.16.2" \
    "pynwb==4.2.0" \
    "pyout==0.8.1" \
    "pyparsing==3.3.2" \
    "python-dateutil==2.9.0.post0" \
    "python-dotenv==1.2.3" \
    "pytz==2025.2" \
    "pyyaml==6.0.3" \
    "referencing==0.37.0" \
    "remfile==0.1.15" \
    "requests==2.32.4" \
    "rfc3339-validator==0.1.4" \
    "rfc3987==1.3.8" \
    "rpds-py==2026.6.3" \
    "ruamel-yaml==0.19.1" \
    "scipy==1.16.3" \
    "secretstorage==3.5.0" \
    "semantic-version==2.10.0" \
    "six==1.17.0" \
    "tenacity==9.1.4" \
    "tensorstore==0.1.85" \
    "tqdm==4.67.3" \
    "typing-extensions==4.16.0" \
    "typing-inspection==0.4.4" \
    "tzdata==2026.3" \
    "uri-template==1.3.0" \
    "urllib3==2.5.0" \
    "wcwidth==0.8.3" \
    "webcolors==25.10.0" \
    "yarl==1.24.5" \
    "zarr==3.1.5" \
    "zarr-checksum==0.4.7"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already loaded in the kernel. Go to **Runtime → Restart session**, then **Run all cells below** (skip this install cell on re-run).

# Neocortical sleep: LFP, units and sleep states in DANDI:000041

[DANDI:000041](https://dandiarchive.org/dandiset/000041) holds the recordings of
Watson BO, Levenstein D, Greene JP, Gelinas JN and Buzsáki G (2016),
[Network homeostasis and state dynamics of neocortical sleep](https://doi.org/10.1016/j.neuron.2016.03.036),
*Neuron* 90(4):839-852. Rats with silicon probes in frontal cortex were left alone in their home cage through
hours of natural sleep and waking.

Each file holds, for one session:

- the 1250 Hz LFP of every probe channel (`processing/ecephys/LFP/lfp`),
- the theta and gamma phase of the LFP on the channels the authors used for UP states, spindles and theta
  (`processing/ecephys/*DecompositionSeries`),
- the waveforms of every detected spike on each shank (`processing/ecephys/SpikeWaveforms<n>`),
- the sorted units, with putative cell type (`units`),
- the sleep states the authors scored (`processing/behavior/sleep_states`).

This notebook streams one session and checks that these streams line up in time: the sleep states against the
spectral content of the LFP, the spikes against the slow oscillation of the LFP, and the units against the spike
waveforms.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO
from scipy.signal import butter, sosfiltfilt, spectrogram

## Stream a session

In [ ]:
dandiset_id = "000041"
asset_path = "sub-BWRat17/sub-BWRat17_ses-BWRat17-121912_ecephys.nwb"

with DandiAPIClient() as client:
    asset = client.get_dandiset(dandiset_id, "draft").get_asset_by_path(asset_path)
    url = asset.get_content_url(follow_redirects=1, strip_query=True)

io = NWBHDF5IO(file=h5py.File(remfile.File(url), "r"), load_namespaces=True)
nwbfile = io.read()
print(nwbfile.session_id, nwbfile.session_start_time.date(), nwbfile.subject.subject_id, nwbfile.subject.strain)

The electrodes are the probe channels, grouped by shank. The `channel_id` column is the channel's number in the
lab's .xml, .dat and .lfp files, and the `*_reference` columns mark the channels the authors used to detect UP
states, spindles and theta.

In [ ]:
electrodes = nwbfile.electrodes.to_dataframe()
electrodes.drop(columns="group").head()

In [ ]:
units = nwbfile.units.to_dataframe()
print(f"{len(units)} units")
units.groupby(["shank", "cell_type"]).size().unstack(fill_value=0)

In [ ]:
sleep_states = nwbfile.processing["behavior"]["sleep_states"].to_dataframe()
sleep_states["duration"] = sleep_states.stop_time - sleep_states.start_time
(sleep_states.groupby("label").duration.sum() / 60).round(1).rename("minutes")

## Do the time bases agree?

All streams share one clock that starts at the beginning of the recording. The LFP is sampled at a fixed rate,
so its span follows from its length; the sleep states, the spikes and the spike waveforms should all fall within
it.

In [ ]:
lfp = nwbfile.processing["ecephys"]["LFP"]["lfp"]
fs = lfp.rate
lfp_duration = lfp.data.shape[0] / fs

spike_times = np.concatenate(units.spike_times.values)
waveform_series = {
    name: series for name, series in nwbfile.processing["ecephys"].data_interfaces.items() if name.startswith("SpikeWaveforms")
}
waveform_times = np.concatenate([series.timestamps[:] for series in waveform_series.values()])

spans = pd.DataFrame(
    {
        "LFP": (lfp.starting_time, lfp.starting_time + lfp_duration),
        "sleep states": (sleep_states.start_time.min(), sleep_states.stop_time.max()),
        "unit spikes": (spike_times.min(), spike_times.max()),
        "spike waveforms": (waveform_times.min(), waveform_times.max()),
    },
    index=["first (s)", "last (s)"],
).T
spans["last (min)"] = (spans["last (s)"] / 60).round(1)
spans

In [ ]:
assert spans["first (s)"].min() >= 0
assert spans["last (s)"].max() <= lfp_duration

## Sleep states against the LFP

The states were scored from the LFP and the EMG, so the LFP should show their signature: slow (delta, 1-4 Hz)
power is high in NREM sleep, and theta (5-10 Hz) power relative to delta is high in REM sleep. If the states were
shifted in time against the LFP, the separation below would blur.

We read the LFP of the spindle reference channel for the whole session and compute its power in 2-second windows.

In [ ]:
reference = "spindle_reference" if "spindle_reference" in electrodes else None
row = int(np.flatnonzero(electrodes[reference])[0]) if reference else 0
print(f"channel {electrodes.channel_id.iloc[row]} on {electrodes.group_name.iloc[row]}")

signal = lfp.data[:, row] * lfp.conversion * 1e6  # microvolts
frequencies, window_starts, power = spectrogram(signal, fs=fs, nperseg=int(2 * fs), noverlap=0)
window_times = lfp.starting_time + window_starts


def band_power(low, high):
    return power[(frequencies >= low) & (frequencies < high)].mean(axis=0)


bands = pd.DataFrame(
    {"time": window_times, "delta": band_power(1, 4), "theta": band_power(5, 10)},
)
bands["theta/delta"] = bands.theta / bands.delta
bands["state"] = "unscored"
for state in sleep_states.itertuples():
    bands.loc[(bands.time >= state.start_time) & (bands.time < state.stop_time), "state"] = state.label

In [ ]:
colors = {"Awake": "tab:orange", "Non-REM": "tab:blue", "REM": "tab:red"}
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True, gridspec_kw=dict(height_ratios=[1, 2, 2]))
for state in sleep_states.itertuples():
    axes[0].axvspan(state.start_time / 3600, state.stop_time / 3600, color=colors[state.label], lw=0)
axes[0].set_yticks([])
axes[0].set_ylabel("state")
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], labels=list(colors), ncol=3, loc="upper right", fontsize=8)
axes[1].semilogy(bands.time / 3600, bands.delta, lw=0.5, color="k")
axes[1].set_ylabel("delta power\n(µV²/Hz)")
axes[2].semilogy(bands.time / 3600, bands["theta/delta"], lw=0.5, color="k")
axes[2].set_ylabel("theta / delta")
axes[2].set_xlabel("time (h)")
plt.tight_layout()

In [ ]:
summary = bands[bands.state != "unscored"].groupby("state")[["delta", "theta/delta"]].median()
summary

In [ ]:
assert summary.delta.idxmax() == "Non-REM"
assert summary["theta/delta"].idxmax() == "REM"

## Spikes against the slow oscillation

In NREM sleep the neocortex alternates between UP states, when neurons fire, and DOWN states, when almost all of
them are silent. DOWN states appear in the LFP as large slow waves, so the population firing rate and the slow
LFP should be tightly coupled, with the strongest (anti-)correlation at a lag near zero. A misaligned spike
clock would move the peak away from zero.

First, a few seconds of NREM sleep on the UP-state reference channel, with the spikes of all units:

In [ ]:
reference = "up_reference" if "up_reference" in electrodes else None
up_row = int(np.flatnonzero(electrodes[reference])[0]) if reference else row
up_signal = signal if up_row == row else lfp.data[:, up_row] * lfp.conversion * 1e6

longest_nrem = sleep_states[sleep_states.label == "Non-REM"].sort_values("duration").iloc[-1]
t0 = longest_nrem.start_time + longest_nrem.duration / 2
window = (t0, t0 + 6)
samples = slice(int((window[0] - lfp.starting_time) * fs), int((window[1] - lfp.starting_time) * fs))
t = lfp.starting_time + np.arange(samples.start, samples.stop) / fs

order = units.sort_values(["cell_type", "shank"]).index
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw=dict(height_ratios=[1, 2]))
axes[0].plot(t, up_signal[samples], color="k", lw=0.7)
axes[0].set_ylabel("LFP (µV)")
for position, unit_id in enumerate(order):
    times = units.spike_times[unit_id]
    times = times[(times >= window[0]) & (times < window[1])]
    color = "tab:blue" if units.cell_type[unit_id] == "inhibitory" else "k"
    axes[1].vlines(times, position, position + 0.8, color=color, lw=0.8)
axes[1].set_ylabel("unit (blue: inhibitory)")
axes[1].set_xlabel("time (s)")
plt.tight_layout()

Now the cross-correlation over all NREM sleep, between the population rate in 20 ms bins and the 0.5-4 Hz LFP.

In [ ]:
bin_size = 0.02  # 25 LFP samples
edges = np.arange(lfp.starting_time, lfp.starting_time + lfp_duration, bin_size)
population_rate = np.histogram(spike_times, edges)[0].astype(float)

slow = sosfiltfilt(butter(3, (0.5, 4), "bandpass", fs=fs, output="sos"), up_signal)
samples_per_bin = int(round(bin_size * fs))
assert np.isclose(samples_per_bin, bin_size * fs), "the bin must hold a whole number of LFP samples"
n_bins = min(len(population_rate), len(slow) // samples_per_bin)
slow_binned = slow[: n_bins * samples_per_bin].reshape(n_bins, samples_per_bin).mean(axis=1)
population_rate = population_rate[:n_bins]
bin_times = edges[:n_bins]

in_nrem = np.zeros(n_bins, bool)
for state in sleep_states[sleep_states.label == "Non-REM"].itertuples():
    in_nrem[(bin_times >= state.start_time) & (bin_times < state.stop_time)] = True


def zscore(x):
    return (x - x[in_nrem].mean()) / x[in_nrem].std()


lags = np.arange(-25, 26)
rate_z, slow_z = zscore(population_rate), zscore(slow_binned)
correlation = np.array([np.mean((rate_z * np.roll(slow_z, -lag))[in_nrem]) for lag in lags])
peak_lag = lags[np.argmax(np.abs(correlation))] * bin_size

plt.figure(figsize=(6, 3))
plt.plot(lags * bin_size * 1000, correlation, color="k")
plt.axvline(0, color="gray", lw=0.5)
plt.xlabel("LFP lag relative to population rate (ms)")
plt.ylabel("correlation")
plt.title(f"NREM sleep; strongest at {peak_lag * 1000:.0f} ms")
plt.tight_layout()

In [ ]:
assert abs(peak_lag) <= 0.1

## Firing rates by state

Most neocortical neurons fire more in waking and REM sleep than in NREM sleep, and inhibitory neurons fire
faster than excitatory ones.

In [ ]:
state_duration = sleep_states.groupby("label").duration.sum()
rates = {}
for unit_id, times in units.spike_times.items():
    counts = {}
    for state in sleep_states.itertuples():
        counts[state.label] = counts.get(state.label, 0) + np.count_nonzero((times >= state.start_time) & (times < state.stop_time))
    rates[unit_id] = {label: counts[label] / state_duration[label] for label in counts}
rates = pd.DataFrame(rates).T
rates["cell_type"] = units.cell_type
rates.groupby("cell_type").median().round(2)

## Units against the spike waveforms

The `SpikeWaveforms<n>` series hold every spike detected on shank *n*, including noise and multi-unit clusters;
the units are the clusters that were sorted as single cells. Every spike of a unit should therefore have a
waveform at exactly the same time on its shank.

In [ ]:
sampling_rate = 20000.0  # wideband sampling rate of the .res spike times
matches = []
for unit_id, unit in units.iterrows():
    shank_times = waveform_series[f"SpikeWaveforms{unit.shank}"].timestamps[:]
    found = np.isin(np.rint(unit.spike_times * sampling_rate), np.rint(shank_times * sampling_rate))
    matches.append(found.mean())
units["fraction_with_waveform"] = matches
print(units.fraction_with_waveform.describe()[["min", "max"]])
assert units.fraction_with_waveform.min() == 1

The mean waveforms of the fastest-firing excitatory and inhibitory units, on the 8 channels of their shank.
Inhibitory (fast-spiking) cells have narrower spikes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, cell_type in zip(axes, ["excitatory", "inhibitory"]):
    candidates = units[units.cell_type == cell_type]
    if not len(candidates):
        ax.set_title(f"no {cell_type} unit")
        continue
    unit = candidates.loc[candidates.spike_times.map(len).idxmax()]
    series = waveform_series[f"SpikeWaveforms{unit.shank}"]
    shank_times = np.rint(series.timestamps[:] * sampling_rate)
    rows = np.flatnonzero(np.isin(shank_times, np.rint(unit.spike_times * sampling_rate)))[:500]
    mean_waveform = series.data[rows].mean(axis=0) * series.conversion * 1e6  # (channels, samples), µV
    n_channels, n_samples = mean_waveform.shape
    sample_times = (np.arange(n_samples) - n_samples // 2) / sampling_rate * 1000
    for channel in range(n_channels):
        ax.plot(sample_times, mean_waveform[channel] - 150 * channel, color="k", lw=1)
    ax.set_title(f"{cell_type} unit {unit.name} (shank {unit.shank}, {len(rows)} spikes)")
    ax.set_xlabel("time (ms)")
axes[0].set_ylabel("µV (channels offset by 150 µV)")
plt.tight_layout()

In [ ]:
io.close()